# 01 — EDA: IdSarcasm Dataset Exploration

Eksplorasi dataset Reddit + Twitter Indonesia Sarcastic dari CSV lokal.

## Prerequisites
```bash
pip install pandas matplotlib seaborn
```
Pastikan sudah jalankan `python scripts/download_data.py` untuk generate CSV di `data/raw/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

RAW_DIR = '../data/raw'

## 1. Load CSV Files

In [ ]:
# Load semua split lalu gabung
reddit_train = pd.read_csv(f'{RAW_DIR}/reddit_train.csv')
reddit_val = pd.read_csv(f'{RAW_DIR}/reddit_validation.csv')
reddit_test = pd.read_csv(f'{RAW_DIR}/reddit_test.csv')
reddit_all = pd.concat([reddit_train, reddit_val, reddit_test], ignore_index=True)

twitter_train = pd.read_csv(f'{RAW_DIR}/twitter_train.csv')
twitter_val = pd.read_csv(f'{RAW_DIR}/twitter_validation.csv')
twitter_test = pd.read_csv(f'{RAW_DIR}/twitter_test.csv')
twitter_all = pd.concat([twitter_train, twitter_val, twitter_test], ignore_index=True)

print('=== Reddit ===')
print(f'  train: {len(reddit_train)} | val: {len(reddit_val)} | test: {len(reddit_test)} | total: {len(reddit_all)}')
print(f'  columns: {reddit_all.columns.tolist()}')

print('\n=== Twitter ===')
print(f'  train: {len(twitter_train)} | val: {len(twitter_val)} | test: {len(twitter_test)} | total: {len(twitter_all)}')
print(f'  columns: {twitter_all.columns.tolist()}')

## 2. Sample Data

In [ ]:
# Reddit: text column = 'text' (bukan 'tweet')
print('=== Reddit sample (5 rows) ===')
print(reddit_all[['text', 'label']].head())
print(f'\nDtypes:\n{reddit_all.dtypes}')

print('\n=== Twitter sample (5 rows) ===')
print(twitter_all[['tweet', 'label']].head())
print(f'\nDtypes:\n{twitter_all.dtypes}')

## 3. Label Distribution (Class Balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Reddit
reddit_labels = reddit_all['label'].value_counts().sort_index()
reddit_labels.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Reddit — Label Distribution')
axes[0].set_xlabel('Label (0=Non-sarcasm, 1=Sarcasm)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Non-sarcasm', 'Sarcasm'], rotation=0)
for i, v in enumerate(reddit_labels):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Twitter
twitter_labels = twitter_all['label'].value_counts().sort_index()
twitter_labels.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Twitter — Label Distribution')
axes[1].set_xlabel('Label (0=Non-sarcasm, 1=Sarcasm)')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Non-sarcasm', 'Sarcasm'], rotation=0)
for i, v in enumerate(twitter_labels):
    axes[1].text(i, v + 100, str(v), ha='center', fontweight='bold')

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Class balance:')
print(f'  Reddit: {reddit_labels.to_dict()} — ratio sarcasm: {reddit_labels.get(1,0)/len(reddit_all):.2%}')
print(f'  Twitter: {twitter_labels.to_dict()} — ratio sarcasm: {twitter_labels.get(1,0)/len(twitter_all):.2%}')

## 4. Text Length Analysis

**Penting:** Reddit pake kolom `text`, Twitter pake kolom `tweet`.

In [ ]:
# Hitung panjang teks — pake kolom yang bener
reddit_all['text_len'] = reddit_all['text'].astype(str).str.len()
twitter_all['text_len'] = twitter_all['tweet'].astype(str).str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reddit
for label, color, name in [(0, '#2ecc71', 'Non-sarcasm'), (1, '#e74c3c', 'Sarcasm')]:
    subset = reddit_all[reddit_all['label'] == label]['text_len']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=name)
axes[0].set_title('Reddit — Text Length Distribution')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Twitter
for label, color, name in [(0, '#2ecc71', 'Non-sarcasm'), (1, '#e74c3c', 'Sarcasm')]:
    subset = twitter_all[twitter_all['label'] == label]['text_len']
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, label=name)
axes[1].set_title('Twitter — Text Length Distribution')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Text Length Stats ===')
print('\nReddit:')
print(reddit_all.groupby('label')['text_len'].describe().round(1))
print('\nTwitter:')
print(twitter_all.groupby('label')['text_len'].describe().round(1))

## 5. Split Distribution

In [ ]:
print('=== Per-Split Label Distribution ===')

print('\nReddit:')
for name, df in [('train', reddit_train), ('val', reddit_val), ('test', reddit_test)]:
    dist = df['label'].value_counts().sort_index().to_dict()
    print(f'  {name}: {len(df)} rows | {dist}')

print('\nTwitter:')
for name, df in [('train', twitter_train), ('val', twitter_val), ('test', twitter_test)]:
    dist = df['label'].value_counts().sort_index().to_dict()
    print(f'  {name}: {len(df)} rows | {dist}')

# Visualisasi split
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, datasets, title in [
    (axes[0], [('train', reddit_train), ('val', reddit_val), ('test', reddit_test)], 'Reddit Split'),
    (axes[1], [('train', twitter_train), ('val', twitter_val), ('test', twitter_test)], 'Twitter Split')
]:
    splits = [name for name, _ in datasets]
    non_sarc = [df['label'].value_counts().get(0, 0) for _, df in datasets]
    sarc = [df['label'].value_counts().get(1, 0) for _, df in datasets]
    x = range(len(splits))
    ax.bar(x, non_sarc, label='Non-sarcasm', color='#2ecc71')
    ax.bar(x, sarc, bottom=non_sarc, label='Sarcasm', color='#e74c3c')
    ax.set_xticks(x)
    ax.set_xticklabels(splits)
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Contoh Sarcasm vs Non-Sarcasm

In [ ]:
TEXT_COL_REDDIT = 'text'
TEXT_COL_TWITTER = 'tweet'

print('=== Twitter — Contoh Sarcasm (label=1) ===')
for i, row in twitter_all[twitter_all['label']==1].head(5).iterrows():
    print(f'  [{i}] {str(row[TEXT_COL_TWITTER])[:150]}')

print('\n=== Twitter — Contoh Non-Sarcasm (label=0) ===')
for i, row in twitter_all[twitter_all['label']==0].head(5).iterrows():
    print(f'  [{i}] {str(row[TEXT_COL_TWITTER])[:150]}')

print('\n=== Reddit — Contoh Sarcasm (label=1) ===')
for i, row in reddit_all[reddit_all['label']==1].head(5).iterrows():
    print(f'  [{i}] {str(row[TEXT_COL_REDDIT])[:150]}')

print('\n=== Reddit — Contoh Non-Sarcasm (label=0) ===')
for i, row in reddit_all[reddit_all['label']==0].head(5).iterrows():
    print(f'  [{i}] {str(row[TEXT_COL_REDDIT])[:150]}')

## 7. Missing Values & Data Quality

In [ ]:
print('=== Missing Values ===')
print('\nReddit:')
print(reddit_all.isnull().sum())
print('\nTwitter:')
print(twitter_all.isnull().sum())

# Cek duplikat
print('\n=== Duplicates ===')
print(f'Reddit text duplicates: {reddit_all["text"].duplicated().sum()}')
print(f'Twitter tweet duplicates: {twitter_all["tweet"].duplicated().sum()}')

## Summary

Catat temuan utama:

| Metric | Reddit | Twitter |
|--------|--------|--------|
| Total samples | | |
| Sarcasm ratio | | |
| Avg text length | | |
| Train/Val/Test | | |
| Duplicates | | |

**Catatan penting:**
- Twitter dataset di HuggingFace hanya ~2.7k (paper claim 12.8k — kemungkinan raw sebelum cleaning/split)
- Reddit kolom teks = `text`, Twitter kolom teks = `tweet`
- Class balance: ~25% sarcasm di kedua dataset (imbalanced)